## Data quality checks

Validate the pipeline before trusting any downstream insight. The cells below check foreign key integrity between facts and dimensions, numeric ranges on percentages and percentiles, table coverage, and the contents of the measure dictionary.

### Importing necessary libraries

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv

# Load Postgres credentials from .env
load_dotenv('../.env')
host = os.getenv('PGHOST', 'localhost')
port = os.getenv('PGPORT', '5432')
db = os.getenv('PGDATABASE', 'ms_health')
user = os.getenv('PGUSER', 'postgres')
pwd = os.getenv('PGPASSWORD', '')
engine = create_engine(f'postgresql+psycopg://{user}:{pwd}@{host}:{port}/{db}')

def q(sql):
    return pd.read_sql(text(sql), engine)

### How many rows ended up in each table?

In [2]:
# Row counts per dimension, fact, and mart table.
q("""
SELECT 'dim_geography' AS table_name, COUNT(*) AS rows FROM dim_geography
UNION ALL SELECT 'dim_measure', COUNT(*) FROM dim_measure
UNION ALL SELECT 'dim_facility', COUNT(*) FROM dim_facility
UNION ALL SELECT 'fact_places', COUNT(*) FROM fact_places
UNION ALL SELECT 'fact_svi', COUNT(*) FROM fact_svi
UNION ALL SELECT 'fact_svi_wide', COUNT(*) FROM fact_svi_wide
UNION ALL SELECT 'fact_acs', COUNT(*) FROM fact_acs
UNION ALL SELECT 'fact_imr', COUNT(*) FROM fact_imr
UNION ALL SELECT 'fact_hospital_quality', COUNT(*) FROM fact_hospital_quality
UNION ALL SELECT 'fact_hpsa_county', COUNT(*) FROM fact_hpsa_county
UNION ALL SELECT 'mart_maternal_risk_index', COUNT(*) FROM mart_maternal_risk_index
UNION ALL SELECT 'mart_drive_time', COUNT(*) FROM mart_drive_time
UNION ALL SELECT 'mart_top20_priority', COUNT(*) FROM mart_top20_priority
ORDER BY 1
""")

,table_name,rows
0,dim_facility,108
1,dim_geography,878
2,dim_measure,67
3,fact_acs,7005
4,fact_hospital_quality,1034
5,fact_hpsa_county,242
6,fact_imr,87
7,fact_places,74201
8,fact_svi,13080
9,fact_svi_wide,875


### Are there any orphan rows in the fact tables?

In [3]:
# LEFT JOIN with WHERE dim IS NULL catches fact rows with no matching dimension.
q("""
SELECT 'fact_places orphan geo_sk' AS check_name, COUNT(*) AS bad_rows
FROM fact_places f
LEFT JOIN dim_geography g ON g.geo_sk = f.geo_sk
WHERE g.geo_sk IS NULL
UNION ALL
SELECT 'fact_places orphan measure_sk', COUNT(*)
FROM fact_places f
LEFT JOIN dim_measure m ON m.measure_sk = f.measure_sk
WHERE m.measure_sk IS NULL
""")

,check_name,bad_rows
0,fact_places orphan measure_sk,0
1,fact_places orphan geo_sk,0


### Are any numeric values outside their expected range?

In [4]:
# Numeric range checks: percentages must lie within 0 to 100, SVI percentiles within 0 to 1.
q("""
SELECT 'fact_places out-of-range percent' AS check_name, COUNT(*) AS bad_rows
FROM fact_places f
JOIN dim_measure m ON m.measure_sk = f.measure_sk
WHERE m.unit = 'percent' AND (f.data_value < 0 OR f.data_value > 100)
UNION ALL
SELECT 'fact_svi_wide out-of-range RPL_THEMES', COUNT(*)
FROM fact_svi_wide WHERE rpl_themes < 0 OR rpl_themes > 1
""")

,check_name,bad_rows
0,fact_places out-of-range percent,0
1,fact_svi_wide out-of-range RPL_THEMES,0


### Did we get the expected coverage?

In [5]:
# Coverage checks: county count, PLACES year count, and hospital geocoding completeness.
q("""
SELECT 'distinct MS counties in dim_geography' AS check_name,
    COUNT(DISTINCT county_fips)::TEXT AS value,
    'expected 82' AS expected
FROM dim_geography
UNION ALL
SELECT 'distinct year_sk in fact_places',
    COUNT(DISTINCT year_sk)::TEXT,
    'expected 4 to 6'
FROM fact_places
UNION ALL
SELECT 'dim_facility missing geom (cannot do spatial query)',
    COUNT(*)::TEXT,
    'expected near 0'
FROM dim_facility WHERE geom IS NULL
""")

,check_name,value,expected
0,distinct MS counties in dim_geography,82,expected 82
1,distinct year_sk in fact_places,7,expected 4 to 6
2,dim_facility missing geom (cannot do spatial q...,3,expected near 0


### Quick look at the measure dictionary

In [6]:
# Measure count per source.
q("SELECT source, COUNT(*) AS measures FROM dim_measure GROUP BY source ORDER BY 1")

,source,measures
0,ACS,8
1,PLACES,44
2,SVI,15
